In [ ]:
# -*- coding: utf-8 -*-
import arcpy
import math
import networkx as nx

arcpy.env.overwriteOutput = True

# ============================================================
# USER SETTINGS
# ============================================================

# --- Inputs
roads_fc_in = r"C:\esriprodata\routing\routing\routing.gdb\roads_ExportFeatures_Project"          # Polyline roads
start_fc_in = r"C:\esriprodata\routing\routing\routing.gdb\start3500"    # Point starts
end_fc_in   = r"C:\esriprodata\routing\routing\routing.gdb\EndPoints_Project"      # Point ends

# --- Output
out_gdb = r"C:\esriprodata\routing\routing\Output.gdb"
out_routes_name = "Routes_AStar_Time_3500"
out_nodes_name  = "GraphNodes_AStar_3500"              # stored nodes for Near snapping

# --- Fields in roads
speed_field  = "maxspeed"     # numeric (km/h)
oneway_field = "Oneway"        # text/int indicating one-way direction

# --- Pairing mode for multiple routes
# "ONE_END"  : all starts route to FIRST end feature
# "BY_ORDER" : pair start[i] to end[i]
# "BY_FIELD" : match starts to ends by common PairID field value
PAIR_MODE  = "ONE_END"     # change if needed: "BY_ORDER" or "BY_FIELD"
PAIR_FIELD = "PairID"      # only used when PAIR_MODE="BY_FIELD"

# --- IDs stored in output (use OID@ or a real field name)
START_ID_FIELD = "OID@"    # e.g. "StartID" or "OID@"
END_ID_FIELD   = "OID@"    # e.g. "EndID"   or "OID@"

# --- Projection handling
AUTO_PROJECT_TO_UTM38N = True
UTM38N_EPSG = 32638  # Riyadh area

# --- Graph building controls
ROUND_XY = 2               # rounding in meters (0.01 m). Adjust if needed.
DEFAULT_SPEED_KMH = 50.0   # used if speed is missing/0
MIN_SPEED_KMH = 5.0        # clamp tiny speeds
MAX_SPEED_KMH = 160.0      # clamp extreme speeds

# Heuristic: lower bound travel time (minutes) = straight distance / max_speed
HEURISTIC_MAX_SPEED_KMH = 120.0

# ============================================================
# HELPERS
# ============================================================

def is_geographic(sr):
    try:
        return sr.type.lower() == "geographic"
    except Exception:
        return False

def round_xy(x, y, ndigits=ROUND_XY):
    return (round(float(x), ndigits), round(float(y), ndigits))

def dist_m(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def heuristic_time_minutes(n1, n2):
    # optimistic (lower bound) time in minutes
    # distance (m) / (max_speed (m/min))
    max_speed_m_per_min = (HEURISTIC_MAX_SPEED_KMH * 1000.0) / 60.0
    return dist_m(n1, n2) / max_speed_m_per_min

def clamp_speed_kmh(v):
    if v is None:
        return DEFAULT_SPEED_KMH
    try:
        v = float(v)
    except Exception:
        return DEFAULT_SPEED_KMH
    if v <= 0:
        return DEFAULT_SPEED_KMH
    return max(MIN_SPEED_KMH, min(MAX_SPEED_KMH, v))

def normalize_oneway(val):
    """
    Returns one of: "BOTH", "FT", "TF"
    FT means geometry direction (from->to) is allowed only
    TF means reverse direction (to->from) is allowed only
    """
    if val is None:
        return "BOTH"
    s = str(val).strip().upper()

    # Common conventions:
    # - "FT" / "TF" (Esri-style)
    # - "B" / "BOTH" / "0" / "N" => both directions
    # - "1" / "Y" / "T" often means one-way but direction unknown -> assume FT
    if s in ("B", "BOTH", "0", "N", "NO", "FALSE", "F"):
        return "BOTH"
    if s in ("FT", "WITH_DIGITIZED", "WITHDIGITIZED", "FWD", "FORWARD"):
        return "FT"
    if s in ("TF", "AGAINST_DIGITIZED", "AGAINSTDIGITIZED", "REV", "REVERSE"):
        return "TF"
    if s in ("1", "Y", "YES", "TRUE", "T"):
        return "FT"  # safest assumption if only says "oneway"
    return "BOTH"

def ensure_output_fc(out_gdb, out_name, sr):
    out_fc = fr"{out_gdb}\{out_name}"
    if not arcpy.Exists(out_fc):
        arcpy.management.CreateFeatureclass(out_gdb, out_name, "POLYLINE", spatial_reference=sr)

    existing = {f.name.lower() for f in arcpy.ListFields(out_fc)}

    def add_if_missing(name, ftype, **kwargs):
        if name.lower() not in existing:
            arcpy.management.AddField(out_fc, name, ftype, **kwargs)
            existing.add(name.lower())

    add_if_missing("Algorithm", "TEXT", field_length=20)
    add_if_missing("StartID", "TEXT", field_length=64)
    add_if_missing("EndID", "TEXT", field_length=64)
    add_if_missing("PairID", "TEXT", field_length=64)

    add_if_missing("TotalMin", "DOUBLE")     # total time minutes
    add_if_missing("TotalLenM", "DOUBLE")    # total length meters

    add_if_missing("StartSnapD", "DOUBLE")
    add_if_missing("EndSnapD", "DOUBLE")
    add_if_missing("NodeCount", "LONG")

    add_if_missing("Status", "TEXT", field_length=30)   # OK / NO_PATH / SNAP_FAIL / NO_MATCH_END / ERROR
    add_if_missing("Msg", "TEXT", field_length=255)

    return out_fc

def create_nodes_fc_from_graph(G, out_gdb, nodes_name, sr):
    nodes_fc = fr"{out_gdb}\{nodes_name}"
    if arcpy.Exists(nodes_fc):
        return nodes_fc

    arcpy.management.CreateFeatureclass(out_gdb, nodes_name, "POINT", spatial_reference=sr)
    arcpy.management.AddField(nodes_fc, "NodeX", "DOUBLE")
    arcpy.management.AddField(nodes_fc, "NodeY", "DOUBLE")

    with arcpy.da.InsertCursor(nodes_fc, ["SHAPE@", "NodeX", "NodeY"]) as ic:
        for (x, y) in G.nodes:
            ic.insertRow([arcpy.PointGeometry(arcpy.Point(x, y), sr), x, y])

    return nodes_fc

def snap_points_to_nodes(in_points_fc, nodes_fc, out_gdb, tmp_name):
    """
    Returns dict keyed by point OID with:
      { oid: {"node": (x,y) or None, "near_dist": float or None, "near_fid": int } }
    """
    tmp_fc = fr"{out_gdb}\{tmp_name}"
    if arcpy.Exists(tmp_fc):
        arcpy.management.Delete(tmp_fc)
    arcpy.management.CopyFeatures(in_points_fc, tmp_fc)

    # Important: location="LOCATION" so NEAR_X/NEAR_Y exist
    arcpy.analysis.Near(tmp_fc, nodes_fc, location="LOCATION", method="PLANAR")

    out = {}
    with arcpy.da.SearchCursor(tmp_fc, ["OID@", "NEAR_X", "NEAR_Y", "NEAR_DIST", "NEAR_FID"]) as cur:
        for oid, nx_, ny_, nd, nf in cur:
            if nx_ is None or ny_ is None or nd is None or nf is None or int(nf) == -1:
                out[oid] = {"node": None, "near_dist": None, "near_fid": int(nf) if nf is not None else -1}
            else:
                out[oid] = {"node": round_xy(nx_, ny_), "near_dist": float(nd), "near_fid": int(nf)}
    return out

def get_first_feature_oid(fc):
    with arcpy.da.SearchCursor(fc, ["OID@"]) as cur:
        for (oid,) in cur:
            return oid
    return None

def safe_str(val):
    if val is None:
        return ""
    return str(val)

def build_polyline_from_nodes(path_nodes, sr):
    arr = arcpy.Array([arcpy.Point(x, y) for (x, y) in path_nodes])
    return arcpy.Polyline(arr, sr)

def compute_totals(G, path_nodes):
    total_min = 0.0
    total_len = 0.0
    for a, b in zip(path_nodes[:-1], path_nodes[1:]):
        data = G[a][b]
        total_min += float(data["weight"])        # minutes
        total_len += float(data.get("len_m", 0.0))
    return total_min, total_len

# ============================================================
# STEP 0: Optional project inputs to UTM 38N if needed
# ============================================================

roads_fc = roads_fc_in
start_fc = start_fc_in
end_fc   = end_fc_in

sr_in = arcpy.Describe(roads_fc_in).spatialReference

if AUTO_PROJECT_TO_UTM38N and is_geographic(sr_in):
    sr_proj = arcpy.SpatialReference(UTM38N_EPSG)

    roads_fc = fr"{out_gdb}\Roads_UTM38N"
    start_fc = fr"{out_gdb}\Start_UTM38N"
    end_fc   = fr"{out_gdb}\End_UTM38N"

    if not arcpy.Exists(roads_fc):
        arcpy.management.Project(roads_fc_in, roads_fc, sr_proj)
    if not arcpy.Exists(start_fc):
        arcpy.management.Project(start_fc_in, start_fc, sr_proj)
    if not arcpy.Exists(end_fc):
        arcpy.management.Project(end_fc_in, end_fc, sr_proj)

sr = arcpy.Describe(roads_fc).spatialReference

# ============================================================
# STEP 1: Build directed graph from roads (A* time cost)
# ============================================================

G = nx.DiGraph()

# Cursor fields: geometry + speed + oneway
fields = ["OID@", "SHAPE@", speed_field, oneway_field]

with arcpy.da.SearchCursor(roads_fc, fields) as cur:
    for oid, geom, spd, ow in cur:
        if geom is None:
            continue

        speed_kmh = clamp_speed_kmh(spd)
        oneway = normalize_oneway(ow)

        # traverse all vertices (very important for connectivity)
        for part in geom:
            pts = [p for p in part if p]
            if len(pts) < 2:
                continue

            for i in range(len(pts) - 1):
                u = round_xy(pts[i].X, pts[i].Y)
                v = round_xy(pts[i+1].X, pts[i+1].Y)

                seg_len_m = dist_m(u, v)
                if seg_len_m <= 0:
                    continue

                # time minutes = distance(m) / speed(m/min)
                speed_m_per_min = (speed_kmh * 1000.0) / 60.0
                seg_min = seg_len_m / speed_m_per_min

                # Add edges based on one-way rule
                def add_edge(a, b):
                    # keep the best (lowest time) if edge repeats
                    if G.has_edge(a, b):
                        if seg_min < G[a][b]["weight"]:
                            G[a][b].update(weight=seg_min, len_m=seg_len_m, road_oid=int(oid), speed_kmh=speed_kmh, oneway=oneway)
                    else:
                        G.add_edge(a, b, weight=seg_min, len_m=seg_len_m, road_oid=int(oid), speed_kmh=speed_kmh, oneway=oneway)

                if oneway == "BOTH":
                    add_edge(u, v)
                    add_edge(v, u)
                elif oneway == "FT":
                    add_edge(u, v)
                elif oneway == "TF":
                    add_edge(v, u)
                else:
                    add_edge(u, v)
                    add_edge(v, u)

print("Graph built.")
print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())

# ============================================================
# STEP 2: Create nodes FC and snap all starts/ends
# ============================================================

nodes_fc = create_nodes_fc_from_graph(G, out_gdb, out_nodes_name, sr)

start_snap = snap_points_to_nodes(start_fc, nodes_fc, out_gdb, "tmp_start_near")
end_snap   = snap_points_to_nodes(end_fc,   nodes_fc, out_gdb, "tmp_end_near")

# ============================================================
# STEP 3: Read start/end rows, build route pairs
# ============================================================

# Build start list
start_fields = ["OID@"] + ([START_ID_FIELD] if START_ID_FIELD != "OID@" else [])
if PAIR_MODE == "BY_FIELD" and PAIR_FIELD not in start_fields:
    start_fields.append(PAIR_FIELD)

starts = []
with arcpy.da.SearchCursor(start_fc, start_fields) as cur:
    for row in cur:
        oid = row[start_fields.index("OID@")]
        sid = safe_str(row[start_fields.index(START_ID_FIELD)]) if START_ID_FIELD != "OID@" else safe_str(oid)
        pid = safe_str(row[start_fields.index(PAIR_FIELD)]) if PAIR_MODE == "BY_FIELD" else ""
        starts.append({"oid": oid, "sid": sid, "pid": pid})

# Build end list
end_fields = ["OID@"] + ([END_ID_FIELD] if END_ID_FIELD != "OID@" else [])
if PAIR_MODE == "BY_FIELD" and PAIR_FIELD not in end_fields:
    end_fields.append(PAIR_FIELD)

ends = []
with arcpy.da.SearchCursor(end_fc, end_fields) as cur:
    for row in cur:
        oid = row[end_fields.index("OID@")]
        eid = safe_str(row[end_fields.index(END_ID_FIELD)]) if END_ID_FIELD != "OID@" else safe_str(oid)
        pid = safe_str(row[end_fields.index(PAIR_FIELD)]) if PAIR_MODE == "BY_FIELD" else ""
        ends.append({"oid": oid, "eid": eid, "pid": pid})

pairs = []

if PAIR_MODE == "ONE_END":
    end_oid = ends[0]["oid"] if len(ends) else None
    if end_oid is None:
        raise RuntimeError("No end features found.")
    for s in starts:
        pairs.append((s["oid"], end_oid, ""))

elif PAIR_MODE == "BY_ORDER":
    n = min(len(starts), len(ends))
    for i in range(n):
        pairs.append((starts[i]["oid"], ends[i]["oid"], ""))

elif PAIR_MODE == "BY_FIELD":
    end_by_pid = {}
    for e in ends:
        end_by_pid.setdefault(e["pid"], []).append(e)
    for s in starts:
        cand = end_by_pid.get(s["pid"], [])
        if cand:
            pairs.append((s["oid"], cand[0]["oid"], s["pid"]))
        else:
            pairs.append((s["oid"], None, s["pid"]))
else:
    raise ValueError("PAIR_MODE must be ONE_END, BY_ORDER, or BY_FIELD")

start_id_by_oid = {s["oid"]: s["sid"] for s in starts}
end_id_by_oid   = {e["oid"]: e["eid"] for e in ends}

# ============================================================
# STEP 4 (UPDATED STEP D): Run A* for each pair & write output
# ============================================================

out_fc = ensure_output_fc(out_gdb, out_routes_name, sr)

insert_fields = [
    "SHAPE@",
    "Algorithm", "StartID", "EndID", "PairID",
    "TotalMin", "TotalLenM",
    "StartSnapD", "EndSnapD", "NodeCount",
    "Status", "Msg"
]

with arcpy.da.InsertCursor(out_fc, insert_fields) as ic:
    for s_oid, e_oid, pid in pairs:
        sid = start_id_by_oid.get(s_oid, safe_str(s_oid))
        eid = end_id_by_oid.get(e_oid, safe_str(e_oid)) if e_oid is not None else ""

        # No matching end in BY_FIELD
        if e_oid is None:
            ic.insertRow([None, "A*_TIME", sid, eid, pid, None, None, None, None, None, "NO_MATCH_END", "No matching end for PairID"])
            continue

        s_info = start_snap.get(s_oid, {})
        e_info = end_snap.get(e_oid, {})

        s_node = s_info.get("node")
        e_node = e_info.get("node")

        if s_node is None or e_node is None:
            ic.insertRow([
                None, "A*_TIME", sid, eid, pid,
                None, None,
                s_info.get("near_dist"), e_info.get("near_dist"),
                None,
                "SNAP_FAIL", "Near did not return a valid snapped node"
            ])
            continue

        try:
            path_nodes = nx.astar_path(
                G,
                s_node,
                e_node,
                heuristic=heuristic_time_minutes,
                weight="weight"
            )

            total_min, total_len = compute_totals(G, path_nodes)
            route_geom = build_polyline_from_nodes(path_nodes, sr)

            ic.insertRow([
                route_geom,
                "A*_TIME", sid, eid, pid,
                total_min, total_len,
                float(s_info["near_dist"]), float(e_info["near_dist"]),
                len(path_nodes),
                "OK", ""
            ])

        except nx.NetworkXNoPath:
            ic.insertRow([
                None, "A*_TIME", sid, eid, pid,
                None, None,
                float(s_info.get("near_dist") or 0.0), float(e_info.get("near_dist") or 0.0),
                None,
                "NO_PATH", "No path between snapped nodes (directed/one-way constraints may block)"
            ])
        except Exception as ex:
            msg = str(ex)[:250]
            ic.insertRow([
                None, "A*_TIME", sid, eid, pid,
                None, None,
                float(s_info.get("near_dist") or 0.0), float(e_info.get("near_dist") or 0.0),
                None,
                "ERROR", msg
            ])

print("Done. Routes written to:", out_fc)
print("Nodes FC:", nodes_fc)

In [1]:
import arcpy
import math
from datetime import datetime, timezone, timedelta

# -----------------------------
# INPUTS
# -----------------------------
out_fc = r"C:\esriprodata\routing\routing\Output.gdb\Routes_AStar_Time_1000"
out_gdb = r"C:\esriprodata\routing\routing\Output.gdb"
out_points_name = "RoutePoints_5m_Time_1000"

STEP_M = 5.0
HEADING_OFFSET_M = 0.5   # metres ahead/behind used to compute bearing; tune if needed

DEPARTURE_ISO_UTC = None  # e.g. "2026-02-25T08:00:00Z"

# -----------------------------
# TIME HELPERS
# -----------------------------
def parse_or_now_utc(iso_z):
    if iso_z:
        return datetime.strptime(iso_z, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
    return datetime.now(timezone.utc)

def to_iso_z(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

route_start_time_utc = parse_or_now_utc(DEPARTURE_ISO_UTC)

# -----------------------------
# HEADING HELPERS
# -----------------------------
def bearing_degrees(x1, y1, x2, y2):
    """
    Returns the bearing in degrees (0–360, clockwise from North)
    between two projected (or geographic) coordinate pairs.
    Works for any CRS; for geographic lat/lon the result is the
    planar approximation which is acceptable for short segments.
    """
    dx = x2 - x1
    dy = y2 - y1
    angle = math.degrees(math.atan2(dx, dy))   # atan2(east, north)
    return angle % 360.0

def cardinal(deg):
    """Convert a 0–360 bearing to one of 8 cardinal / intercardinal labels."""
    dirs = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    idx = int((deg + 22.5) / 45.0) % 8
    return dirs[idx]

def heading_at_distance(geom, d, route_len, offset=HEADING_OFFSET_M):
    """
    Estimate travel heading at distance d along geom by sampling
    a point slightly behind and a point slightly ahead.
    Returns (heading_deg, cardinal_str).
    """
    d_back  = max(d - offset, 0.0)
    d_ahead = min(d + offset, route_len)

    # If both collapsed to the same spot (very short route), fall back wider
    if abs(d_ahead - d_back) < 1e-9:
        d_back  = 0.0
        d_ahead = route_len

    pt_back  = geom.positionAlongLine(d_back,  use_percentage=False).firstPoint
    pt_ahead = geom.positionAlongLine(d_ahead, use_percentage=False).firstPoint

    hdg = bearing_degrees(pt_back.X, pt_back.Y, pt_ahead.X, pt_ahead.Y)
    return hdg, cardinal(hdg)

# -----------------------------
# OUTPUT FC SETUP
# -----------------------------
sr = arcpy.Describe(out_fc).spatialReference
out_pts_fc = fr"{out_gdb}\{out_points_name}"

if arcpy.Exists(out_pts_fc):
    arcpy.management.Delete(out_pts_fc)

arcpy.management.CreateFeatureclass(out_gdb, out_points_name, "POINT", spatial_reference=sr)

arcpy.management.AddField(out_pts_fc, "RouteOID",    "LONG")
arcpy.management.AddField(out_pts_fc, "Algorithm",   "TEXT", field_length=20)
arcpy.management.AddField(out_pts_fc, "StartID",     "TEXT", field_length=64)
arcpy.management.AddField(out_pts_fc, "EndID",       "TEXT", field_length=64)
arcpy.management.AddField(out_pts_fc, "PairID",      "TEXT", field_length=64)
arcpy.management.AddField(out_pts_fc, "Status",      "TEXT", field_length=30)

arcpy.management.AddField(out_pts_fc, "TotalMin",    "DOUBLE")
arcpy.management.AddField(out_pts_fc, "TotalLenM",   "DOUBLE")

arcpy.management.AddField(out_pts_fc, "StepM",       "DOUBLE")
arcpy.management.AddField(out_pts_fc, "CumDistM",    "DOUBLE")
arcpy.management.AddField(out_pts_fc, "CumMin",      "DOUBLE")

arcpy.management.AddField(out_pts_fc, "TimeISO",     "TEXT",   field_length=20)
arcpy.management.AddField(out_pts_fc, "TimeUTC",     "DATE")

# ── NEW ──────────────────────────────────────────────────────────────────────
arcpy.management.AddField(out_pts_fc, "Heading",     "DOUBLE")          # 0–360 °, CW from North
arcpy.management.AddField(out_pts_fc, "CardinalDir", "TEXT", field_length=4)   # N / NE / E …
# ─────────────────────────────────────────────────────────────────────────────

# -----------------------------
# GENERATE POINTS
# -----------------------------
route_fields = [
    "OID@", "SHAPE@",
    "Algorithm", "StartID", "EndID", "PairID",
    "Status",
    "TotalMin", "TotalLenM"
]

insert_fields = [
    "SHAPE@",
    "RouteOID", "Algorithm", "StartID", "EndID", "PairID",
    "Status",
    "TotalMin", "TotalLenM",
    "StepM", "CumDistM", "CumMin",
    "TimeISO", "TimeUTC",
    "Heading", "CardinalDir"      # ← NEW
]

def safe_float(v):
    try:
        return float(v)
    except Exception:
        return None

with arcpy.da.SearchCursor(out_fc, route_fields) as rcur, \
     arcpy.da.InsertCursor(out_pts_fc, insert_fields) as icur:

    for (roid, geom, alg, sid, eid, pid, status, total_min, total_len_m) in rcur:

        if geom is None or status != "OK":
            continue

        route_len = float(geom.length)
        if route_len <= 0:
            continue

        total_min = safe_float(total_min)

        num_steps = int(route_len // STEP_M)
        distances = [i * STEP_M for i in range(num_steps + 1)]
        if abs(distances[-1] - route_len) > 1e-6:
            distances.append(route_len)

        for d in distances:
            pt_geom = geom.positionAlongLine(d, use_percentage=False)

            # Time
            if total_min is not None:
                frac       = d / route_len
                cum_min    = frac * total_min
                t          = route_start_time_utc + timedelta(minutes=cum_min)
                time_iso   = to_iso_z(t)
                time_utc_date = t.replace(tzinfo=None)
            else:
                cum_min       = None
                time_iso      = ""
                time_utc_date = None

            # Heading  ← NEW
            hdg, card = heading_at_distance(geom, d, route_len)

            icur.insertRow([
                pt_geom,
                int(roid), str(alg), str(sid), str(eid), str(pid),
                str(status),
                safe_float(total_min), safe_float(total_len_m),
                float(STEP_M), float(d), safe_float(cum_min),
                time_iso, time_utc_date,
                hdg, card           # ← NEW
            ])

print("Done.")
print("Created points feature class:", out_pts_fc)
print("Route start time (UTC):", to_iso_z(route_start_time_utc))

Done.
Created points feature class: C:\esriprodata\routing\routing\Output.gdb\RoutePoints_5m_Time_1000
Route start time (UTC): 2026-03-16T10:10:18Z


In [ ]:
import arcpy
from datetime import datetime, timezone, timedelta

# -----------------------------
# INPUTS
# -----------------------------
out_fc = r"C:\esriprodata\routing\routing\Output.gdb\Routes_AStar_Time_3500"   # your routes output (polyline)
out_gdb = r"C:\esriprodata\routing\routing\Output.gdb"
out_points_name = "RoutePoints_5m_Time_3500"

STEP_M = 5.0

# Optional: set a fixed departure time in UTC ISO8601 (Z), or leave None to use "now"
DEPARTURE_ISO_UTC = None  # e.g. "2026-02-25T08:00:00Z"

# -----------------------------
# TIME HELPERS
# -----------------------------
def parse_or_now_utc(iso_z):
    if iso_z:
        # expects "YYYY-MM-DDThh:mm:ssZ"
        return datetime.strptime(iso_z, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
    return datetime.now(timezone.utc)

def to_iso_z(dt):
    # Ensure UTC and format with trailing Z
    dt_utc = dt.astimezone(timezone.utc)
    return dt_utc.strftime("%Y-%m-%dT%H:%M:%SZ")

route_start_time_utc = parse_or_now_utc(DEPARTURE_ISO_UTC)

# -----------------------------
# OUTPUT FC SETUP
# -----------------------------
sr = arcpy.Describe(out_fc).spatialReference
out_pts_fc = fr"{out_gdb}\{out_points_name}"

if arcpy.Exists(out_pts_fc):
    arcpy.management.Delete(out_pts_fc)

arcpy.management.CreateFeatureclass(out_gdb, out_points_name, "POINT", spatial_reference=sr)

# Copy / store “all details” you likely care about at point-level
arcpy.management.AddField(out_pts_fc, "RouteOID", "LONG")
arcpy.management.AddField(out_pts_fc, "Algorithm", "TEXT", field_length=20)
arcpy.management.AddField(out_pts_fc, "StartID", "TEXT", field_length=64)
arcpy.management.AddField(out_pts_fc, "EndID", "TEXT", field_length=64)
arcpy.management.AddField(out_pts_fc, "PairID", "TEXT", field_length=64)
arcpy.management.AddField(out_pts_fc, "Status", "TEXT", field_length=30)

arcpy.management.AddField(out_pts_fc, "TotalMin", "DOUBLE")
arcpy.management.AddField(out_pts_fc, "TotalLenM", "DOUBLE")

arcpy.management.AddField(out_pts_fc, "StepM", "DOUBLE")
arcpy.management.AddField(out_pts_fc, "CumDistM", "DOUBLE")  # distance along route
arcpy.management.AddField(out_pts_fc, "CumMin", "DOUBLE")     # minutes since route start

# Store timestamp as text ISO 8601 Z (as requested)
arcpy.management.AddField(out_pts_fc, "TimeISO", "TEXT", field_length=20)

# Optional: also store as a real Date field (handy in GIS time sliders)
arcpy.management.AddField(out_pts_fc, "TimeUTC", "DATE")

# -----------------------------
# GENERATE POINTS ALONG EACH ROUTE (manual, accurate, with attributes)
# -----------------------------
route_fields = [
    "OID@", "SHAPE@",
    "Algorithm", "StartID", "EndID", "PairID",
    "Status",
    "TotalMin", "TotalLenM"
]

insert_fields = [
    "SHAPE@",
    "RouteOID", "Algorithm", "StartID", "EndID", "PairID",
    "Status",
    "TotalMin", "TotalLenM",
    "StepM", "CumDistM", "CumMin",
    "TimeISO", "TimeUTC"
]

def safe_float(v):
    try:
        return float(v)
    except Exception:
        return None

with arcpy.da.SearchCursor(out_fc, route_fields) as rcur, \
     arcpy.da.InsertCursor(out_pts_fc, insert_fields) as icur:

    for (roid, geom, alg, sid, eid, pid, status, total_min, total_len_m) in rcur:

        # Only generate points for successful routes with geometry
        if geom is None or status != "OK":
            continue

        # Prefer geometry length as truth for spacing
        route_len = float(geom.length)
        if route_len <= 0:
            continue

        total_min = safe_float(total_min)
        if total_min is None:
            # If TotalMin missing, you can still create points, but CumMin/time will be null
            total_min = None

        # If TotalLenM exists but differs slightly, that’s okay; we use geom.length for spacing.
        # Time interpolation uses route_len as denominator to align with point placement.
        num_steps = int(route_len // STEP_M)

        # Ensure we include the end point exactly
        distances = [i * STEP_M for i in range(num_steps + 1)]
        if abs(distances[-1] - route_len) > 1e-6:
            distances.append(route_len)

        for d in distances:
            pt_geom = geom.positionAlongLine(d, use_percentage=False)

            # Compute time at this point by proportional distance along route
            if total_min is not None:
                frac = d / route_len
                cum_min = frac * total_min
                t = route_start_time_utc + timedelta(minutes=cum_min)
                time_iso = to_iso_z(t)
                time_utc_date = t.replace(tzinfo=None)  # ArcGIS DATE is naive local/UTC; we store UTC naive
            else:
                cum_min = None
                time_iso = ""
                time_utc_date = None

            icur.insertRow([
                pt_geom,
                int(roid), str(alg), str(sid), str(eid), str(pid),
                str(status),
                safe_float(total_min), safe_float(total_len_m),
                float(STEP_M), float(d), safe_float(cum_min),
                time_iso, time_utc_date
            ])

print("Done.")
print("Created points feature class:", out_pts_fc)
print("Route start time (UTC):", to_iso_z(route_start_time_utc))